### Какие данные и признаки я использовала и почему
Для решения задачи я опиралась на комбинацию текстовых и категориальных признаков, так как текстовые запросы в сфере услуг часто бывают короткими, а намерение пользователя сильно зависит от региона.
- Исторические клики (Train): Главный источник данных. Если пользователи ранее массово выбирали конкретные item_id по запросу «автоподбор», этот сигнал является самым надежным предиктором.
- Текстовые признаки (item_title_raw, item_description_raw, параметры): Объединены в единое текстовое поле. При этом заголовок объявления дублируется дважды, это сделано намеренно, чтобы повысить его вес в поиске по сравнению с описанием, которое часто содержит избыточную воду или SEO-спам.
- Локация и Категория (location_id, category_id): Ключевые признаки для услуг. Они были добавлены к текстовым векторам с заданными вручную весами (локация получила вес 2.0, категория 1.0), чтобы совпадение города давало мощный буст (boost) релевантным объявлениям.

### Какие модели или алгоритмы применены
Архитектура решения состоит из каскада двух этапов:
- Click Graph (Исторический поиск): Алгоритм группирует обучающую выборку по приведенному к нижнему регистру тексту запроса и собирает топ-50 самых популярных (кликабельных) объявлений. На инференсе для знакомых запросов мы в первую очередь отдаем эти проверенные варианты.
- Векторный поиск (TF-IDF + Metadata):
    - Если запрос новый (проблема «холодного старта») или исторических кликов не хватает для заполнения топ-50, запускается контентный поиск.
    - Используется TfidfVectorizer (ngram_range=(1,2), max_features=100000).
    - Категориальные фичи кодируются через OneHotEncoder.
    - Текстовые и OHE-векторы склеиваются в единую разреженную матрицу (через scipy.sparse.hstack).
Кандидаты отбираются через быстрое умножение матриц (косинусное сходство) с последующим использованием np.argpartition для оптимизации скорости сортировки.

### Как проверялось качество решения до отправки
Для локальной проверки (без траты попыток на платформе) была реализована Hold-out валидация.
- Я разбивала train.parquet на обучающую и валидационную выборку.
- Чтобы избежать утечки данных (data leakage), разбиение делалось по уникальным сессиям поиска (комбинация search_query + search_location_id).
- Около 80% сессий использовались для построения графа кликов и обучения TF-IDF, а оставшиеся 20% выступали в роли локального бенчмарка (Ground Truth), на котором считалась целевая метрика Recall@50. Это позволило подобрать оптимальные веса для признаков (например, вес 2.0 для локации).

### Какие типы ошибок найдены при анализе и что с ними сделано
- Ошибка Scikit-learn (ValueError при OneHotEncoding): Модель падала при применении энкодера на тестовых данных, так как названия колонок локации и категории в трейне (item_location_id) и бенчмарке (search_location_id) отличались. Решение: Передача в .transform() сырых numpy-массивов с помощью .values, чтобы энкодер ориентировался на данные, а не на названия колонок.
- Шум в описаниях (SEO-тексты): Обычный TF-IDF часто выводил в топ нерелевантные объявления просто из-за того, что в их длинном описании много раз повторялось искомое слово. Решение: Ручное дублирование заголовка (item_title_raw + ' ' + item_title_raw) при конкатенации текста. Это сместило фокус алгоритма на самую суть объявления.
- Дубликаты в кандидатах: При комбинировании исторического и векторного поиска одни и те же item_id могли попадать в итоговый список дважды, съедая драгоценные слоты (которых всего 50). Решение: Добавлен строгий фильтр (if item not in candidates), а векторный поиск отбирает кандидатов "с запасом" (топ-60), чтобы после отсева дублей гарантированно добить размер ответа до 50.

In [1]:
import pandas as pd
import numpy as np
import re
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix
import warnings

warnings.filterwarnings('ignore')

TRAIN_PATH = 'train.parquet'
TEST_QUERIES_PATH = 'benchmark_queries.parquet'
TEST_ITEMS_PATH = 'benchmark_items.parquet'

In [2]:
print("Загрузка данных")
train_df = pd.read_parquet(TRAIN_PATH)
queries_df = pd.read_parquet(TEST_QUERIES_PATH)
items_df = pd.read_parquet(TEST_ITEMS_PATH)

print(f"Train size: {train_df.shape}")
print(f"Benchmark queries size: {queries_df.shape}")
print(f"Benchmark items size: {items_df.shape}")

Загрузка данных
Train size: (497673, 19)
Benchmark queries size: (2452, 6)
Benchmark items size: (189212, 14)


In [3]:
print("Создаем словарь популярных кликов из train")

# Группируем по тексту запроса и считаем частоту кликов на каждый item_id
# Используем текст запроса в нижнем регистре для унификации
train_df['query_lower'] = train_df['search_query'].str.lower().str.strip()

# Считаем популярность (количество раз, когда объявление было выбрано по этому запросу)
click_counts = train_df.groupby(['query_lower', 'item_id']).size().reset_index(name='clicks')
click_counts = click_counts.sort_values(['query_lower', 'clicks'], ascending=[True, False])

# Создаем словарь: текст запроса -> список отсортированных по популярности item_id
historical_clicks = {}
for query, group in tqdm(click_counts.groupby('query_lower'), desc="Сбор словаря"):
    # Берем топ-50 самых кликабельных объявлений для запроса
    historical_clicks[query] = group['item_id'].tolist()[:50]
    
print(f"Найдено уникальных запросов в истории: {len(historical_clicks)}")

Создаем словарь популярных кликов из train


Сбор словаря:   0%|          | 0/74529 [00:00<?, ?it/s]

Найдено уникальных запросов в истории: 74529


In [4]:
print("Предобработка текстов")

def clean_text(text):
    if pd.isna(text):
        return ""
    # Оставляем только буквы и цифры
    text = re.sub(r'[^a-zA-Zа-яА-Я0-9\s]', ' ', str(text))
    # Убираем лишние пробелы и приводим к нижнему регистру
    return re.sub(r'\s+', ' ', text).strip().lower()

# Готовим тексты для корпуса объявлений (Items)
# Дублируем title дважды для повышения его веса в поиске
items_df['text_feature'] = (
    items_df['item_title_raw'].fillna('') + ' ' + 
    items_df['item_title_raw'].fillna('') + ' ' + 
    items_df['item_infm_params_text'].fillna('') + ' ' + 
    items_df['item_description_raw'].fillna('')
).apply(clean_text)

# Готовим тексты для запросов (Queries)
queries_df['text_feature'] = (
    queries_df['search_query'].fillna('') + ' ' + 
    queries_df['search_infm_params_text'].fillna('')
).apply(clean_text)

Предобработка текстов


In [5]:
print("Векторизация текстов (TF-IDF)...")
# Настраиваем TF-IDF. Ограничим словарь (max_features), чтобы не тратить лишнюю память, 
# но оставим достаточно для хорошего полнотекстового поиска
tfidf = TfidfVectorizer(max_features=100000, analyzer='word', ngram_range=(1, 2))

# Обучаем на текстах объявлений и трансформируем их
items_text_vectors = tfidf.fit_transform(items_df['text_feature'])
queries_text_vectors = tfidf.transform(queries_df['text_feature'])

print("Векторизация категорий и локаций...")
# Локация имеет огромный вес в услугах. Если локация поиска и объявления совпадают, 
# векторное произведение должно давать мощный буст.

# ДОБАВЛЕНО .values, чтобы убрать названия колонок и избежать ValueError
item_locs = items_df[['item_location_id']].fillna('-1').astype(str).values
query_locs = queries_df[['search_location_id']].fillna('-1').astype(str).values

item_cats = items_df[['item_category_id']].fillna('-1').astype(str).values
query_cats = queries_df[['search_category']].fillna('-1').astype(str).values

# One-hot encoding для локаций
loc_encoder = OneHotEncoder(handle_unknown='ignore')
items_loc_vectors = loc_encoder.fit_transform(item_locs)
queries_loc_vectors = loc_encoder.transform(query_locs)

# One-hot encoding для категорий
cat_encoder = OneHotEncoder(handle_unknown='ignore')
items_cat_vectors = cat_encoder.fit_transform(item_cats)
queries_cat_vectors = cat_encoder.transform(query_cats)

# Объединяем текстовые и мета-признаки в единую разреженную матрицу.
# Зададим веса (эмпирические). Локация очень важна!
WEIGHT_TEXT = 1.0
WEIGHT_LOC = 2.0  # Буст за совпадение локации
WEIGHT_CAT = 1.0  # Буст за совпадение категории

items_final_vectors = hstack([
    items_text_vectors * WEIGHT_TEXT,
    items_loc_vectors * WEIGHT_LOC,
    items_cat_vectors * WEIGHT_CAT
]).tocsr()

queries_final_vectors = hstack([
    queries_text_vectors * WEIGHT_TEXT,
    queries_loc_vectors * WEIGHT_LOC,
    queries_cat_vectors * WEIGHT_CAT
]).tocsr()

print(f"Размерность итоговой матрицы корпуса: {items_final_vectors.shape}")

Векторизация текстов (TF-IDF)...
Векторизация категорий и локаций...
Размерность итоговой матрицы корпуса: (189212, 102924)


In [6]:
print("Генерация кандидатов")

# Создадим быстрый маппинг индекса матрицы в item_id
idx_to_itemid = items_df['item_id'].values
query_ids = queries_df['query_id'].values
search_queries_raw = queries_df['search_query'].str.lower().str.strip().values

predictions = []

# Чтобы не словить OutOfMemory, считаем косинусное сходство по одному запросу
# Matrix multiplication @ для разреженных матриц работает очень быстро
for i in tqdm(range(len(queries_df)), desc="Поиск по запросам"):
    q_id = query_ids[i]
    q_text = search_queries_raw[i]
    
    candidates = []
    
    # 1. Сначала берем исторические клики (если есть)
    if q_text in historical_clicks:
        candidates.extend(historical_clicks[q_text])
        
    # 2. Если кандидатов меньше 50, добиваем векторным поиском
    if len(candidates) < 50:
        # Умножаем вектор 1-го запроса на матрицу всех айтемов
        # Вектора не нормализованы до 1, но dot product отлично ранжирует
        scores = queries_final_vectors[i].dot(items_final_vectors.T).toarray()[0]
        
        # Находим индексы топ-60 объявлений (с запасом, чтобы отсеять дубликаты из истории)
        # argpartition работает быстрее, чем полный argsort
        top_indices = np.argpartition(scores, -60)[-60:]
        # Сортируем топ-60 по убыванию скора
        top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
        
        # Добавляем в кандидаты
        for idx in top_indices:
            item = idx_to_itemid[idx]
            if item not in candidates: # Избегаем повторов
                candidates.append(item)
            if len(candidates) >= 50:
                break
                
    # Обрезаем ровно до 50
    predictions.append(candidates[:50])

Генерация кандидатов


Поиск по запросам:   0%|          | 0/2452 [00:00<?, ?it/s]

In [7]:
print("Сохранение результатов")

# Преобразуем списки кандидатов в строки, разделенные пробелом
answer_strings = [' '.join(top50) for top50 in predictions]

# Создаем DataFrame
answer_df = pd.DataFrame({
    'query_id': query_ids,
    'answer': answer_strings
})

# Проверки перед сохранением (sanity checks)
assert answer_df.shape[0] == 2452, "Неверное количество запросов"
assert answer_df.shape[1] == 2, "Должно быть ровно две колонки"
assert 'query_id' in answer_df.columns and 'answer' in answer_df.columns, "Неверные названия колонок"

# Сохраняем в csv
answer_df.to_csv('answer.csv', index=False, encoding='utf-8')

print("answer.csv готов")

Сохранение результатов
answer.csv готов
